In [1]:
# WARNING будет загружен датасет и сами модели
!dvc pull

Everything is up to date.


# Результаты

Работа с трансформерными моделями оказалась сложнее чем предполагалось. Библиотека transformers скрывает почти все под капот из-за чего очень тяжело понять что не так. Сообщество не настолько большое чтобы возникающие проблемы было легко решать.

В целом понятно, но непонятно.
MLFlow так же помогает отслеживать запуски. Наверное лучший инструмент для работы с ML.

Немного оффтопа и ниже результаты. Для запуска кода в ячейке ниже нужно получить метрики, которые скачиваются из интернета (???). То есть для базового Accuracy нужен интернет. Причем не простой, а с VPN иначе не скачается. С каждым днем мы все дальше от бога.

RuBERT и T5 на подборе гиперпараметров показали разные результаты. Последние из которых:
- RuBERT: 83%
- T5: 74%

Результаты запуска GPT. Подбором количества примеров удалось достичь точности в 63%.

![img](ruGpt3.png)



In [2]:
import evaluate
import torch
from datasets import DownloadMode
from transformers.pipelines.pt_utils import KeyDataset

from export_import_model import import_model
from model_utils import get_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Нужно VPN подключение для загрузки метрик
accuracy = evaluate.load("accuracy", download_mode=DownloadMode.REUSE_CACHE_IF_EXISTS)
f1 = evaluate.load("f1", download_mode=DownloadMode.REUSE_CACHE_IF_EXISTS)
for model_type in ["ai-forever-ruBert-base", "ai-forever-ruT5-base"]:
    pipeline = import_model(model_type)

    dataset, max_length = get_dataset()
    preds = []
    for out in pipeline(KeyDataset(dataset["dev"], "sentence"), max_length=max_length):
        preds.append(1 if out["label"] == "LABEL_1" else 0)

    print(model_type, "metrics:")
    print(accuracy.compute(predictions=preds, references=dataset["dev"]["acceptable"]))
    print(f1.compute(predictions=preds, references=dataset["dev"]["acceptable"]))

    del pipeline
    del preds


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ai-forever-ruBert-base metrics:
{'accuracy': 0.8151207115628971}
{'f1': 0.8837395125848981}


Loading weights:   0%|          | 0/263 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.shared.weight to transformer.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie transformer.shared.weight to transformer.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


ai-forever-ruT5-base metrics:
{'accuracy': 0.7312579415501906}
{'f1': 0.8444280985656492}
